## Setup


In [ ]:
# Install packages
# %pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math

In [ ]:
def code_break():
    print("-----------------------------------------")

def get_ssr(matrix: np.ndarray, estimators: np.array):
    ""

In [ ]:
apple = pd.read_csv("APPLE.csv")
apple.head()

## Question 3 ? Manual coefficient estimates


## Estimators

In [ ]:
# 3. 

X = apple[['regprc', 'ecoprc', 'inseason']]
X = sm.add_constant(X)
y = apple['ecolbs']

# ----- Manual Calc ------
X = np.array(X)
XtX = X.T @ X

print(f"XtX: {XtX}")

det_XtX = np.linalg.det(XtX)
# display(f"Determinant XtX: {det_XtX}")
inv_XtX = np.linalg.inv(XtX)

print(f"inv_XtX: {inv_XtX}")

# code_break()

Xty = X.T @ y
# print(Xty)

# code_break()

estimators = inv_XtX @ Xty
# print(estimators)

# estimators_results = zip(['const', 'x1', 'x2', 'x3'], estimators)
estimators_results = pd.DataFrame(estimators, index = ['const', 'x1', 'x2', 'x3'])
estimators_results.rename(mapper={0:'manual'}, axis=1, inplace=True)
estimators_results

## Question 4 ? Homoskedastic covariance and standard errors


## Standard Errors

In [ ]:
# Manual Calculation
beta = np.asarray(estimators).reshape(-1)
e = np.asarray(y).reshape(-1) - X @ beta
e_2 = e**2
ssr = e_2.sum()

s_2 = ssr / (X.shape[0] - X.shape[1])

cov_matrix = s_2*(inv_XtX)
print(cov_matrix)

In [ ]:
manual_cov_matrix = pd.DataFrame(
    cov_matrix, 
    index=['const', 'regprc', 'ecoprc', 'inseason'],
    columns=['const', 'regprc', 'ecoprc', 'inseason']
)
display(manual_cov_matrix)

## Question 5 ? Compare with regression output


In [ ]:
model1 = sm.OLS(exog=X, endog=y)
results = model1.fit()

results.summary()

In [ ]:
stats_package_est = pd.DataFrame(results.params)
# comparison_table.reset_index(inplace=True)
stats_package_est.rename(mapper={0:'stats_package'}, inplace=True, axis=1)
stats_package_est

comparison_table = pd.merge(stats_package_est, estimators_results, left_index=True, right_index=True)

display(comparison_table)

In [ ]:
sp_cov_matrix = results.cov_params()
for i in range(0, 2):
    sp_cov_matrix.rename(mapper={
        'const':'const',
        'x1':'regprc',
        'x2':'ecoprc',
        'x3':'inseason',
    },
    axis=i,
    inplace=True
    )

sp_cov_matrix

## Question 6 ? FWL theorem


## Question 7 ? Model (2) and coefficient interpretation


## Model 2

In [ ]:
X = apple[['ecoprc', 'regprc', 'reglbs', 'age', 'faminc', 'hhsize', 'numlt5']]
X = sm.add_constant(X)
y = apple['ecolbs']

In [ ]:
model2 = sm.OLS(exog=X, endog=y)
results = model2.fit()

results.summary()

## Question 8 ? Own-price hypothesis test and confidence interval


## Question 9 ? HC2 covariance and revised inference


## HC2 Errors

In [ ]:
## Diagonal
X_np = np.asarray(X)
inv_XtX = np.linalg.inv(X_np.T @ X_np)
projection_matrix = X_np @ inv_XtX @ X_np.T
# print(projection_matrix)
# code_break()
# print(projection_matrix.shape)
# code_break()
px_diag = projection_matrix.diagonal()

## residuals
u2 = np.asarray((results.resid)**2)

print(type(px_diag),type(u2))
print(px_diag.shape, u2.shape)

## Omega
omega_weights = np.asarray(results.resid)**2 / (1 - px_diag)

## HC2 Errors
XtX = X.T @ X
inv_XtX = np.linalg.inv(XtX)

hc2_cov = inv_XtX @ (X_np.T * omega_weights) @ X_np @ inv_XtX
code_break()
pd.DataFrame(hc2_cov, columns=model2.exog_names, index=model2.exog_names)

In [ ]:
pd.DataFrame(results.cov_HC2, columns=model2.exog_names, index=model2.exog_names)

## Question 10 ? Joint F test for the D variables


## Unassigned draft cells


## Test, ecolbs >= 0, ecolbs < 0